# 50 Startups Profit Prediction
### CRISP-DM Methodology — Multiple Linear Regression

**Objective:** Predict startup profit based on R&D, Administration, Marketing spends and State.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('whitegrid')

## 1. Data Understanding

In [ ]:
df = pd.read_csv('../data/50_Startups.csv')
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:\n{df.isnull().sum()}')
df.head(10)

In [ ]:
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0,0].scatter(df['R&D Spend'], df['Profit'], alpha=0.6)
axes[0,0].set_xlabel('R&D Spend'); axes[0,0].set_ylabel('Profit')
axes[0,0].set_title('R&D Spend vs Profit')
z = np.polyfit(df['R&D Spend'], df['Profit'], 1)
p = np.poly1d(z)
axes[0,0].plot(df['R&D Spend'], p(df['R&D Spend']), 'r--', alpha=0.8)

axes[0,1].scatter(df['Administration'], df['Profit'], alpha=0.6)
axes[0,1].set_xlabel('Administration'); axes[0,1].set_ylabel('Profit')
axes[0,1].set_title('Administration vs Profit')
z2 = np.polyfit(df['Administration'], df['Profit'], 1)
p2 = np.poly1d(z2)
axes[0,1].plot(df['Administration'], p2(df['Administration']), 'r--', alpha=0.8)

axes[0,2].scatter(df['Marketing Spend'], df['Profit'], alpha=0.6)
axes[0,2].set_xlabel('Marketing Spend'); axes[0,2].set_ylabel('Profit')
axes[0,2].set_title('Marketing Spend vs Profit')
z3 = np.polyfit(df['Marketing Spend'], df['Profit'], 1)
p3 = np.poly1d(z3)
axes[0,2].plot(df['Marketing Spend'], p3(df['Marketing Spend']), 'r--', alpha=0.8)

axes[1,0].hist(df['Profit'], bins=15, edgecolor='black', alpha=0.7)
axes[1,0].axvline(df['Profit'].mean(), color='r', linestyle='--', label=f'Mean: {df["Profit"].mean():.0f}')
axes[1,0].axvline(df['Profit'].median(), color='g', linestyle='--', label=f'Median: {df["Profit"].median():.0f}')
axes[1,0].legend(); axes[1,0].set_title('Profit Distribution')

df.boxplot(column=['R&D Spend','Administration','Marketing Spend','Profit'], ax=axes[1,1])
axes[1,1].set_title('Boxplots for Outlier Detection')
axes[1,1].tick_params(axis='x', rotation=45)

df['State'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[1,2])
axes[1,2].set_title('State Distribution')

plt.tight_layout()
plt.show()

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title('Correlation Matrix')
plt.show()
print(corr)

In [ ]:
sns.pairplot(df, hue='State', diag_kind='kde')
plt.show()

## 3. Data Preparation

In [ ]:
X = df.drop('Profit', axis=1)
y = df['Profit']

ct = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(drop='first'), ['State'])],
    remainder='passthrough'
)
X_encoded = ct.fit_transform(X)

encoded_cols = ct.named_transformers_['encoder'].get_feature_names_out(['State'])
feature_names = list(encoded_cols) + ['R&D Spend', 'Administration', 'Marketing Spend']
print(f'Encoded features: {feature_names}')
print(f'Encoded shape: {X_encoded.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} samples')
print(f'Test: {X_test.shape[0]} samples')

## 4. Model Training

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f'Intercept: {model.intercept_:,.4f}')
for name, coef in zip(feature_names, model.coef_):
    print(f'{name}: {coef:,.4f}')

## 5. Model Evaluation

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print(f"{'Metric':<20} {'Train':<15} {'Test':<15}")
print('-' * 50)
print(f"{'R² Score':<20} {r2_score(y_train, y_train_pred):<15.4f} {r2_score(y_test, y_test_pred):<15.4f}")
print(f"{'MAE':<20} {mean_absolute_error(y_train, y_train_pred):<15,.2f} {mean_absolute_error(y_test, y_test_pred):<15,.2f}")
print(f"{'RMSE':<20} {np.sqrt(mean_squared_error(y_train, y_train_pred)):<15,.2f} {np.sqrt(mean_squared_error(y_test, y_test_pred)):<15,.2f}")

In [ ]:
residuals = y_test - y_test_pred
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_pred, residuals, alpha=0.6)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Profit'); axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual Plot')

from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': model.coef_})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)
print(coef_df)

print(f'\nMost influential: {coef_df.iloc[0]["Feature"]}')

## 7. Conclusion

- R&D Spend is the most important predictor of Profit
- Marketing Spend is the secondary factor
- Administration has minimal impact
- State (location) has limited influence
- Model achieves high R², suitable for deployment